In [ ]:
# %pip install -q pymupdf pytesseract pillow pandas

# ============================================================
# Invoice extraction
# The invoices are scanned images (PDFs with no text layer, and PNG screenshots), so text is read by OCR.
# OCR (Tesseract) + layout-aware rules. Handles the layouts seen so far:
# classic "From / Bill To", letterhead-at-top, and EDI "LABEL: value" invoices.
# Output: pandas DataFrame `df` with the 7 required columns.
# Setup:  pip install pymupdf pytesseract pillow pandas
#         plus the Tesseract binary (apt-get install -y tesseract-ocr  /  brew install tesseract)
# ============================================================
import glob, io, os, re
from datetime import datetime

import pandas as pd
import pymupdf                      # renders PDFs to images
import pytesseract
from PIL import Image

INVOICE_DIR = "Invoices"            # folder with the PDFs / images

COLUMNS = ["vendor_name", "vendor_address", "payment_terms", "invoice_value",
           "company_code", "po_reference", "invoice_id"]


# ---------- 1. Load any invoice (PDF or image) as page images ----------
def load_pages(path, dpi=200):
    if path.lower().endswith(".pdf"):
        doc = pymupdf.open(path)
        return [Image.open(io.BytesIO(p.get_pixmap(dpi=dpi).tobytes("png"))).convert("RGB") for p in doc]
    img = Image.open(path).convert("RGB")
    if img.width < 1500:            # upscale small screenshots for better OCR
        img = img.resize((img.width * 2, img.height * 2), Image.LANCZOS)
    return [img]


# ---------- 2. OCR + layout-aware rules ----------
def ocr_segments(img):
    """OCR words, grouped into line segments; a big horizontal gap splits two columns."""
    d = pytesseract.image_to_data(img, config="--psm 3", output_type=pytesseract.Output.DICT)
    gap = img.width * 0.04
    lines = {}
    for i, w in enumerate(d["text"]):
        if w.strip() and float(d["conf"][i]) > 0:
            key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
            lines.setdefault(key, []).append((d["left"][i], d["top"][i], d["width"][i], w))
    segs = []
    for words in lines.values():
        words.sort()
        cur = [words[0]]
        for w in words[1:]:
            if w[0] - (cur[-1][0] + cur[-1][2]) > gap:
                segs.append(cur); cur = [w]
            else:
                cur.append(w)
        segs.append(cur)
    out = [{"x": s[0][0], "y": min(w[1] for w in s), "text": " ".join(w[3] for w in s)} for s in segs]
    return sorted(out, key=lambda s: (s["y"], s["x"]))

def _after_label(text, labels, pattern, window=250):
    for lab in labels:
        m = re.search(lab, text, re.I)
        if m:
            v = re.search(pattern, text[m.end():m.end() + window], re.I)
            if v:
                return v.group(1)
    return None

def _on_same_row(segs, label, pattern, img_h):
    """Value printed on the same visual row as a label (e.g. right-aligned amounts)."""
    for lab in (s for s in segs if re.search(label, s["text"], re.I)):
        inline = re.search(pattern, lab["text"][re.search(label, lab["text"], re.I).end():], re.I)
        if inline:
            return inline.group(1)
        row = [s for s in segs if s is not lab and abs(s["y"] - lab["y"]) < img_h * 0.006 and s["x"] > lab["x"]]
        for s in sorted(row, key=lambda s: s["x"]):
            v = re.search(pattern, s["text"], re.I)
            if v:
                return v.group(1)
    return None

def _block_below(segs, start, img_w, stop=r"phone|email|e-mail|tel|fax|@"):
    """Address lines below a segment, in the same column."""
    lines = []
    for s in segs:
        if s["y"] > start["y"] and abs(s["x"] - start["x"]) < img_w * 0.05:
            if re.search(stop, s["text"], re.I) or len(lines) >= 4:
                break
            lines.append(s["text"])
    return lines

def extract_ocr(pages):
    img = pages[0]
    text = "\n".join(pytesseract.image_to_string(p, config="--psm 3") for p in pages)
    segs = ocr_segments(img)
    rec = {}

    rec["invoice_id"] = (re.search(r"\bINV[-\w]*\d", text) or [None])[0]
    rec["po_reference"] = _after_label(text, [r"PO[\s_]*REF\w*", r"Purchase\s*Order"], r"(\d{5,11})")
    rec["company_code"] = (re.search(r"CompanyCode\s?\d+", text) or [None])[0]
    rec["payment_terms"] = (_after_label(text, [r"PAYMENT[\s_]*TERMS\s*:"], r"(net\s*\d+\s*days?)", 40)
                            or (re.search(r"due within \d+ days", text, re.I) or [None])[0])
    money = r"€?\s*([\d.,]+\.\d{2})"
    total_label = r"(?<![A-Za-z])TOTAL[\s_]*(INVOICE[\s_]*)?AMOUNT|^Total\b"   # not "Subtotal"
    rec["invoice_value"] = (_on_same_row(segs, total_label, money, img.height)
                            or _after_label(text, [r"(?<![A-Za-z])TOTAL[\s_]*(INVOICE[\s_]*)?AMOUNT",
                                                   r"\bTotal\b"], money, 60))
    if not rec["invoice_value"]:                     # derive gross total from subtotal and VAT rate
        net = (_on_same_row(segs, r"SUB\s*_?\s*TOTAL", money, img.height)
               or _after_label(text, [r"SUB\s*_?\s*TOTAL"], money, 60))
        vat = re.search(r"(?:VAT|Tax)\D{0,6}(\d{1,2})\s*%", text, re.I)
        if net and vat:
            rec["invoice_value"] = round(to_number(net) * (1 + int(vat.group(1)) / 100), 2)
    if not rec["payment_terms"]:                     # derive from due date - invoice date
        rec["payment_terms"] = _days_from_dates(text)

    # vendor name + address: EDI label, else block under "From", else letterhead
    name = _after_label(text, [r"SUPPLIER_NAME\s*:"], r"([^\n]+)", 80)
    if name:
        addr = [_after_label(text, [r"SUPPLIER_NAME"], rf"{lab}\s*:\s*([^\n]+)", 400)
                for lab in ("ADDRESS_LINE1", "CITY_STATE_ZIP")]
        rec["vendor_name"], rec["vendor_address"] = name.strip(), ", ".join(a.strip() for a in addr if a)
    else:
        hdr = next((s for s in segs if re.fullmatch(r"(invoice )?from:?", s["text"], re.I)), None)
        if hdr:
            below = _block_below(segs, hdr, img.width)
            rec["vendor_name"] = below[0] if below else None
            rec["vendor_address"] = ", ".join(below[1:])
        else:
            top = next(s for s in segs if not re.search(r"invoice|INV-", s["text"], re.I))
            rec["vendor_name"] = top["text"]
            rec["vendor_address"] = ", ".join(_block_below(segs, top, img.width))
    return rec

def _days_from_dates(text):
    dates = []
    for lab in (r"INVOICE[\s_]*DATE|\bDate\b", r"DUE[\s_]*DATE"):
        raw = _after_label(text, [lab], r"([A-Z][a-z]+ \d{1,2}, \d{4}|\d{8}|\d{4}-\d{2}-\d{2})", 60)
        for fmt in ("%B %d, %Y", "%Y%m%d", "%Y-%m-%d"):
            try:
                dates.append(datetime.strptime(raw, fmt)); break
            except (TypeError, ValueError):
                pass
    return f"{(dates[1] - dates[0]).days} days" if len(dates) == 2 else None


# ---------- 3. Normalise to clean values ----------
def to_number(v):
    if v is None or isinstance(v, (int, float)):
        return v
    s = re.sub(r"[^\d.,-]", "", str(v))
    if "," in s and "." in s:
        s = s.replace(",", "") if s.rfind(".") > s.rfind(",") else s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".") if len(s.split(",")[-1]) == 2 else s.replace(",", "")
    return float(s) if s else None

def to_days(v):
    m = re.search(r"(\d+)\s*(?:days?)?", str(v or ""), re.I)
    return int(m.group(1)) if m else None

def normalise(rec):
    po = re.sub(r"\D", "", str(rec.get("po_reference") or ""))
    return {
        "vendor_name": (rec.get("vendor_name") or "").strip() or None,
        "vendor_address": re.sub(r"\s+", " ", rec.get("vendor_address") or "").strip(" ,") or None,
        "payment_terms": to_days(rec.get("payment_terms")),        # days, e.g. 30
        "invoice_value": to_number(rec.get("invoice_value")),      # gross, incl. VAT
        "company_code": (re.search(r"CompanyCode\s?\d+", str(rec.get("company_code") or ""))
                         or [rec.get("company_code")])[0],
        "po_reference": po.zfill(11) if po else None,              # 11 chars, left-padded
        "invoice_id": rec.get("invoice_id"),
    }


# ---------- 4. Run over every invoice in the folder ----------
records = []
files = sorted(f for f in glob.glob(os.path.join(INVOICE_DIR, "*"))
               if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff")))
for path in files:
    records.append(normalise(extract_ocr(load_pages(path))))

df = pd.DataFrame(records, columns=COLUMNS)

# ---------- 5. Quality checks ----------
assert df["po_reference"].str.len().eq(11).all(), "po_reference must be 11 characters"
missing = df[df.isna().any(axis=1)]
if not missing.empty:
    print("Rows with missing fields - please review:\n", missing)
df

